<a href="https://colab.research.google.com/github/joaoslino50/kl-hybrid-gcr/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph-Constrained Reasoning for Reliable LLM Question Answering (`kl-hybrid-gcr`)

**Google Colab Runner** — Open directly from GitHub (`File -> Open notebook -> GitHub -> joaoslino50/kl-hybrid-gcr`).

In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive so outputs survive Colab disconnects
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/kl-hybrid-gcr-outputs"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# 2. Clone or update the repository
if not os.path.exists("/content/kl-hybrid-gcr"):
    !git clone https://github.com/joaoslino50/kl-hybrid-gcr.git
%cd /content/kl-hybrid-gcr
!git pull origin main
!pip install -q -r requirements.txt

Mounted at /content/drive
Cloning into 'kl-hybrid-gcr'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 149 (delta 63), reused 131 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.12 MiB | 22.09 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/kl-hybrid-gcr
From https://github.com/joaoslino50/kl-hybrid-gcr
 * branch            main       -> FETCH_HEAD
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
from getpass import getpass
from huggingface_hub import login

# Prompts you with a hidden password box so your token is never saved in plain text in the .ipynb file
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your HF_TOKEN: ").strip()

login(token=os.environ["HF_TOKEN"])
!python scripts/verify_setup.py

Paste your HF_TOKEN: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GCR-PCST-KGQA Backbone Setup Verification

[1/3] Checking core package versions...
  - python                : 3.13.16
  - torch                 : 2.14.1+cu130
  - transformers          : 5.22.1
  - sentence_transformers : 5.7.0
  - networkx              : 3.7
  - pcst_fast             : pcst_fast (loaded)

[2/3] Detecting compute device and running matrix multiplication...
2026-11-17 18:36:04,142 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA L4)
  - Active device         : cuda
  - MatMul output shape   : (128, 128) (norm=1448.4880)

[3/3] Testing SubgraphData, NetworkX conversion, KGTrie & KGTrieLogitsProcessor...
  - SubgraphData -> DiGraph: 4 nodes, 3 edges
  - Allowed next tokens   : [20, 25]
  - Unmasked logit tokens : [20, 25] (all invalid tokens masked to -inf)

SUCCESS: All environment, device, and KG-Trie interface checks passed!


## Checkpoint 1 Evaluations (PCST Retrieval, Baselines, and Live Wikidata Demo)

In [3]:
# 1. PCST Retrieval evaluation on 200 WebQSP samples (SentenceBERT + pcst_fast)
!python scripts/eval_retrieval.py \
    --source webqsp \
    --max-samples 200 \
    --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/retrieval_webqsp_200.jsonl

PCST params: topk_nodes=50 topk_edges=50 edge_cost=0.2 anchor_prize=4.0
README.md: 100% 900/900 [00:00<00:00, 3.55MB/s]
modules.json: 100% 349/349 [00:00<00:00, 1.57MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 537kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 4.21MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 224kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.83MB/s]

model.safetensors: downloading bytes:  78% 70.5M/90.9M [00:01<00:00, 68.6MB/s, 2.96MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 53.0MB/s, 8.14MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 56.7MB/s, 8.77MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 7041.67it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.61MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 56.0MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 87.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 557kB/s]
config.json: 100% 190/190 [00:00

In [4]:
# 2. Crash-safe Checkpoint 1 baselines on 100 WebQSP samples (Llama-3-8B)
!python scripts/eval_baselines.py --mode llm_only_zero_shot --max-samples 100 --resume --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/llm_zs_100.jsonl
!python scripts/eval_baselines.py --mode llm_only_cot --max-samples 100 --resume --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/llm_cot_100.jsonl
!python scripts/eval_baselines.py --mode standard_rag --max-samples 100 --resume --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/rag_100.jsonl

Device: cuda | mode: llm_only_zero_shot | source: webqsp/test
config.json: 100% 654/654 [00:00<00:00, 2.73MB/s]
tokenizer_config.json: 100% 51.0k/51.0k [00:00<00:00, 19.1MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:01<00:00, 8.47MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 374kB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 42.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/11.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  14% 2.19G/16.1G [00:09<01:04, 216MB/s,  237MB/s  ]
Reconstructing (incomplete total...):  17% 2.66G/16.1G [00:09<00:28, 462MB/s,  216MB/s  ]
Reconstructing (incomplete total...):  24% 3.81G/16.1G [00:12<00:39, 310MB/s,  408MB/s  ]

Fetching 4 files:  

In [5]:
# 3. Live Wikidata SPARQL acquisition + Hybrid pipeline + Interactive HTML graph export
!git pull origin main
!python scripts/run_pipeline.py \
    --source wikidata \
    --mode hybrid \
    --export-html /content/drive/MyDrive/kl-hybrid-gcr-outputs/wikidata_demo.html

2026-11-17 19:07:02,961 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA L4)
Traceback (most recent call last):
  File "/content/kl-hybrid-gcr/scripts/run_pipeline.py", line 369, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/kl-hybrid-gcr/scripts/run_pipeline.py", line 158, in main
    subgraphs = acquire_subgraphs(
        source=source,
    ...<2 lines>...
        use_mock=args.use_mock_models,
    )
  File "/content/kl-hybrid-gcr/scripts/run_pipeline.py", line 135, in acquire_subgraphs
    client.fetch_l_hop_subgraph(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        question_id="wikidata_001",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        use_mock=use_mock,
        ^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/kl-hybrid-gcr/src/acquisition/sparql_client.py", line 164, in fetch_l_hop_subgraph
    resolved = self.resolve_entity_qid(entity)
  File "/content/kl-hybrid-gcr/src/acquisition/sparql_client.py", line 66, in 

## Checkpoint 2 / Full Hybrid vs. All-Modes Pipeline

In [6]:
!python scripts/run_pipeline.py \
    --source webqsp \
    --mode hybrid \
    --max-samples 5 \
    --output-json /content/drive/MyDrive/kl-hybrid-gcr-outputs/webqsp_hybrid_5.json \
    --export-html /content/drive/MyDrive/kl-hybrid-gcr-outputs/webqsp_sample0.html

2026-11-17 19:07:10,453 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA L4)
2026-11-17 19:07:10,453 | INFO | src.acquisition.dataset_loader | Loading dataset 'rmanluo/RoG-webqsp' (split='test', streaming=True)
2026-11-17 19:07:10,741 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-11-17 19:07:10,747 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rmanluo/RoG-webqsp/c0632533135a06f8c5d536b420deec5fcb5c58f3/README.md?%2Fdatasets%2Frmanluo%2FRoG-webqsp%2Fresolve%2Fmain%2FREADME.md=&etag=%22ffc0eb5b1125d866311eddac4f8d339a37c87cd0%22 "HTTP/1.1 200 OK"
2026-11-17 19:07:10,991 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py "HTTP/1.1 404 Not Found"
2026-11-17 19:07:11,688 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datas

In [7]:
!python scripts/run_pipeline.py \
    --source webqsp \
    --mode all \
    --max-samples 5 \
    --output-json /content/drive/MyDrive/kl-hybrid-gcr-outputs/webqsp_all_modes_5.json

2026-11-17 19:08:00,781 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA L4)
2026-11-17 19:08:00,781 | INFO | src.acquisition.dataset_loader | Loading dataset 'rmanluo/RoG-webqsp' (split='test', streaming=True)
2026-11-17 19:08:01,061 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-11-17 19:08:01,068 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rmanluo/RoG-webqsp/c0632533135a06f8c5d536b420deec5fcb5c58f3/README.md?%2Fdatasets%2Frmanluo%2FRoG-webqsp%2Fresolve%2Fmain%2FREADME.md=&etag=%22ffc0eb5b1125d866311eddac4f8d339a37c87cd0%22 "HTTP/1.1 200 OK"
2026-11-17 19:08:01,312 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py "HTTP/1.1 404 Not Found"
2026-11-17 19:08:02,005 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datas

In [8]:
from google.colab import runtime
print("Experiment complete! Disconnecting A100 runtime to save Compute Units...")
runtime.unassign()

Experiment complete! Disconnecting A100 runtime to save Compute Units...
